# pyjmri exploration notebook

This notebook walks through the four things you most often want to do interactively against a running JMRI server:

1. Open a `Client` and discover the layout.
2. Change a specific turnout by system name.
3. Acquire several locomotives at once.
4. Send commands to each locomotive independently, in any order, from any cell.

Run the cells top-to-bottom once. After that the `jmri`, `layout`, and throttle handles (`t1`, `t2`, `t3`) stay bound for the rest of the session, so you can re-run any command cell as often as you like.

**Target**: `192.168.1.159:12080` is the layout machine in the basement. Commands sent here actually move trains and throw turnouts.

## 1. Imports and logging

`TurnoutState` is the enum used for `CLOSED` / `THROWN`. The logging config writes WebSocket traffic to `pyjmri.log` next to the notebook; switch to `level=logging.INFO` if the DEBUG output is too noisy.

In [1]:
import logging
from pyjmri import Client, TurnoutState

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    filename="pyjmri.log",
    force=True,
)

## 2. Open the client and discover the layout

`Client.__aenter__` opens the HTTP and WebSocket connections and keeps them open for the rest of the notebook. `jmri.discover()` returns a `Layout` object whose collections (`turnouts`, `sensors`, `blocks`, ...) are populated from JMRI's current state.

Re-running this cell on an already-open client raises `RuntimeError` — run the **shutdown** cell at the bottom first if you need to reconnect.

In [3]:
jmri = await Client("192.168.1.159:12080").__aenter__()
layout = await jmri.discover()
print(f"discovered: {len(layout.turnouts)} turnouts, {len(layout.sensors)} sensors, {len(layout.blocks)} blocks")

discovered: 52 turnouts, 63 sensors, 38 blocks


## 3. Change a specific turnout

Look the turnout up by system name (`NT102`, `NT100`, ...) or by user name if one is assigned. `set_state` sends the command to JMRI; remember that NCE is open-loop, so the state JMRI reports is *the last commanded state*, not an observed one.

In [14]:
turnout = layout.turnouts.by_system_name("NT102")
print(f"name={turnout.name}  user_name={turnout.user_name}  current state={turnout.state.name}")

name=NT102  user_name=South Turnout 102  current state=THROWN


In [9]:
await layout.turnouts.by_system_name("NT102").set_state(TurnoutState.CLOSED)

In [12]:
await layout.turnouts.by_system_name("NT102").set_state(TurnoutState.THROWN)

## 4. Acquire several locomotives

`layout.throttle(dcc_address, long=...)` returns a `Throttle` that is an async context manager. The `async with` form is convenient for one-shot use inside a single cell, but it releases the throttle as soon as the block ends — no good if you want to drive the loco from later cells.

To keep a throttle alive across cells, enter the context manually with `__aenter__()` and bind it to a variable. Each throttle has its own WebSocket correlation and heartbeat task, so the three handles below are fully independent.

Use `long=True` for DCC addresses ≥ 128, `long=False` for short addresses.

In [18]:
t1 = await layout.throttle(2570, long=True).__aenter__()
t2 = await layout.throttle(8096, long=True).__aenter__()
t3 = await layout.throttle(5488,   long=True).__aenter__()
print("acquired three throttles")

acquired three throttles


## 5. Drive each locomotive independently

Every cell below acts on exactly one throttle. Run them in any order, re-run them as often as you like — the throttle handles persist until you release them.

Speed is a float from `0.0` (stop) to `1.0` (full). `forward=False` reverses direction.

In [ ]:
await t1.set_speed(0.4, forward=True)

In [ ]:
await t2.set_speed(0.3, forward=True)

In [26]:
await t3.set_speed(0.1, forward=False)

In [ ]:
await t1.set_speed(0.0, forward=True)

In [ ]:
await t2.set_speed(0.0, forward=False)

In [27]:
await t3.set_speed(0.0, forward=True)

In [ ]:
t3

## 6. Run multiple locomotives concurrently

There is only one event loop in the notebook, but you can run as many *tasks* on it as you like. Two patterns cover almost everything:

- **`asyncio.gather(...)`** — start several coroutines and wait for them all to finish. The cell unblocks when the slowest one is done.
- **`asyncio.create_task(...)`** — start a coroutine in the background and return immediately, so you can keep working in other cells. Join later with `await task`, or abort with `task.cancel()`.

Plain sequential `await run_for(t1, ...)` followed by `await run_for(t2, ...)` does **not** overlap — the second call only starts once the first finishes. Use `gather` or `create_task` whenever you want true concurrency.

The helper below puts a loco at a target speed, sleeps, then forces it back to 0 — even if the task is cancelled mid-run.

In [19]:
import asyncio

async def run_for(throttle, speed, seconds, forward=True):
    try:
        await throttle.set_speed(speed, forward=forward)
        await asyncio.sleep(seconds)
    finally:
        await throttle.set_speed(0.0, forward=forward)

### Pattern A — `asyncio.gather`: run t1 for 10 s and t2 for 5 s in parallel

Both locos start at the same instant; t2 stops after 5 s, t1 keeps going until 10 s, and the cell returns once the longer run finishes.

In [20]:
await asyncio.gather(
    run_for(t1, 0.2, 10),
    run_for(t2, 0.2, 10),
    run_for(t3, 0.2, 10)
)

[None, None, None]

### Pattern B — `asyncio.create_task`: start the runs in the background

The cell returns immediately, so you can keep issuing commands (or watch sensor traffic) while the trains move. Later cells can `await task1` to join, or `task1.cancel()` to abort cleanly — the `finally` in `run_for` guarantees the throttle is set back to 0.

In [23]:
task1 = asyncio.create_task(run_for(t1, 0.1, 10))
task2 = asyncio.create_task(run_for(t2, 0.1, 5))
task3 = asyncio.create_task(run_for(t3, 0.1, 10))
print("both tasks running in background")

both tasks running in background


In [24]:
await asyncio.gather(task1, task2)
print("both tasks finished")

both tasks finished


In [25]:
await asyncio.gather(task3)

[None]

In [ ]:
for task in (task1, task2):
    if not task.done():
        task.cancel()
print("any running tasks cancelled")

## 7. Shutdown

Stop every loco, release each throttle, then close the client. Releasing throttles before closing the client avoids leaving orphaned throttle sessions on the JMRI side. Restarting the kernel also cleans everything up if you forget.

In [28]:
for t in (t1, t2, t3):
    try:
        await t.set_speed(0.0, forward=True)
        await t.release()
    except Exception as e:
        print(f"release failed (probably already released): {e}")
print("throttles released")

throttles released


In [31]:
await jmri.__aexit__(None, None, None)
print("client closed")

client closed
